# Linear and Logistic Regression

**Objective:** Demonstrate linear and logistic regression using domain-style datasets and evaluate model performance.

**Dataset:** Diabetes regression and breast cancer classification datasets

This notebook is Colab-ready and saves tables, metrics, and visual outputs under
`results/`. Public datasets or compact sample datasets are used so the workflow
remains reproducible.


In [ ]:
!pip install -q pandas numpy matplotlib seaborn scikit-learn


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, mean_squared_error, r2_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)


In [ ]:
diabetes = load_diabetes(as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(diabetes.data, diabetes.target, test_size=0.25, random_state=42)
linear = LinearRegression().fit(X_train, y_train)
pred_reg = linear.predict(X_test)

cancer = load_breast_cancer(as_frame=True)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(cancer.data, cancer.target, test_size=0.25, random_state=42, stratify=cancer.target)
scaler = StandardScaler()
Xc_train_scaled = scaler.fit_transform(Xc_train)
Xc_test_scaled = scaler.transform(Xc_test)
logistic = LogisticRegression(max_iter=1000).fit(Xc_train_scaled, yc_train)
pred_cls = logistic.predict(Xc_test_scaled)
proba = logistic.predict_proba(Xc_test_scaled)[:, 1]

metrics = pd.DataFrame(
    [
        {"model": "Linear Regression", "metric": "RMSE", "value": mean_squared_error(y_test, pred_reg, squared=False)},
        {"model": "Linear Regression", "metric": "R2", "value": r2_score(y_test, pred_reg)},
        {"model": "Logistic Regression", "metric": "Accuracy", "value": accuracy_score(yc_test, pred_cls)},
        {"model": "Logistic Regression", "metric": "F1", "value": f1_score(yc_test, pred_cls)},
        {"model": "Logistic Regression", "metric": "AUC", "value": roc_auc_score(yc_test, proba)},
    ]
)
metrics.to_csv(RESULTS_DIR / "regression_metrics.csv", index=False)
display(metrics)


In [ ]:
predictions = pd.DataFrame({"actual": y_test, "predicted": pred_reg})
predictions.to_csv(RESULTS_DIR / "linear_predictions.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(data=predictions, x="actual", y="predicted", ax=axes[0])
axes[0].set_title("Linear Regression: Actual vs Predicted")
axes[0].grid(alpha=0.25)
cls_metrics = metrics[metrics["model"] == "Logistic Regression"]
sns.barplot(data=cls_metrics, x="metric", y="value", ax=axes[1])
axes[1].set_ylim(0, 1)
axes[1].set_title("Logistic Regression Metrics")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "regression_dashboard.png", dpi=180)
plt.show()
